# U-Net Image Segmentation Model Architecture Explaination.

### 1. ```DoubleConv``` class

This module defines a basic building block used repeatedly in both the encoder and decoder paths.

In [5]:
import torch.nn as nn

In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


#### Step-by-step explanation:
- The block performs two convolutional operations per call.

- Each layer consists of:

   1. A 3×3 convolution (padding = 1 ensures output size = input size).

   2. A Batch Normalization layer for stabilizing training.

   3. A ReLU activation (in-place for memory efficiency).

- It repeats this pattern twice, which increases representational power.

- This design helps extract local image features while preserving spatial resolution.

For example, DoubleConv(64, 128) converts a 64-channel input into a 128-channel feature map with the same height and width.

### 2. ```UNet``` class
The main architecture combines contracting (encoder) and expansive (decoder) paths.

In [3]:
class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()


### Encoder (Contracting Path)

In [7]:
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super(UNet, self).__init__()

        # Encoder (Contracting Path)
        self.inc = DoubleConv(n_channels, 64)

        self.down1 = nn.MaxPool2d(2)
        self.conv1 = DoubleConv(64, 128)

        self.down2 = nn.MaxPool2d(2)
        self.conv2 = DoubleConv(128, 256)

        self.down3 = nn.MaxPool2d(2)
        self.conv3 = DoubleConv(256, 512)

        self.down4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)


In [8]:
model = UNet(n_channels=3, n_classes=1)


This path reduces spatial dimensions while increasing channel depth:

- Each ```MaxPool2d(2)``` halves height and width.

- Each ```DoubleConv``` doubles the channels (64→128→256→512→1024).

- The bottleneck (middle) is the most compressed representation.

### Decoder (Expansive Path)

In [9]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


In [11]:
class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super(UNet, self).__init__()


        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv5 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv6 = DoubleConv(256, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv7 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, n_classes, 1)


The decoder upsamples the feature maps back to the original image size:

- Each ```ConvTranspose2d``` doubles the spatial resolution.

- After upsampling, it concatenates with the corresponding encoder feature map (skip connection).

- Then ```DoubleConv``` refines merged features.

Skip connections give access to encoder’s high-resolution features that were lost due to pooling — improving localization.

### 3. ```forward()``` method

This defines how data flows through the network.

In [12]:
def forward(self, x):
    # Encoder
    x1 = self.inc(x)                      # [B, 64, H, W]
    x2 = self.conv1(self.down1(x1))       # [B, 128, H/2, W/2]
    x3 = self.conv2(self.down2(x2))       # [B, 256, H/4, W/4]
    x4 = self.conv3(self.down3(x3))       # [B, 512, H/8, W/8]
    x5 = self.bottleneck(self.down4(x4))  # [B, 1024, H/16, W/16]

    # Decoder
    x = self.up1(x5)                      # [B, 512, H/8, W/8]
    x = torch.cat([x, x4], dim=1)         # Concatenate skip (512+512)
    x = self.conv4(x)

    x = self.up2(x)                       # [B, 256, H/4, W/4]
    x = torch.cat([x, x3], dim=1)
    x = self.conv5(x)

    x = self.up3(x)                       # [B, 128, H/2, W/2]
    x = torch.cat([x, x2], dim=1)
    x = self.conv6(x)

    x = self.up4(x)                       # [B, 64, H, W]
    x = torch.cat([x, x1], dim=1)
    x = self.conv7(x)

    return torch.sigmoid(self.outc(x))

#### Key details:
- ```torch.cat(..., dim=1)``` concatenates feature maps across channel dimension.

- The final ```Conv2d(64, n_classes, 1)``` reduces channels to the segmentation output (e.g., 1 channel for binary segmentation).

- ```torch.sigmoid()``` squashes output between 0–1 → interpretable as pixel-wise probability (foreground class probability).

### Overall Architecture Summary

| Stage   | Operation                     | Output Channels | Spatial Size Change |
| ------- | ----------------------------- | --------------- | ------------------- |
| Input   | Image                         | 3               | H × W               |
| Encoder | Downsampling + Conv Blocks    | 64–1024         | ↓ (by 16× total)    |
| Decoder | Upsampling + Skip Connections | 512–64          | ↑ (back to H × W)   |
| Output  | 1×1 Conv + Sigmoid            | 1 (mask)        | Same as input       |